# Black Friday — desconto nos títulos em atraso

Oferta de 20% de desconto na quitação à vista até 28/11/2026, para os clientes marcados
**Sim** na aba `resumo_` da planilha *Black Friday (Inadimplência)*. O valor de cada cliente
é o saldo vencido em aberto **hoje no DW**; os e-mails vêm do cadastro do CIGAM no DW.

Envio por `contato.financeiro@ctasmart.com.br`. Repositório:
https://github.com/yagoppereira/comunicacoes_financeiras

**Ordem:** 1 → 2 → 3 → 4 → 5 → 6 → 7 (teste) → 8 (produção). Nada é enviado antes da seção 7.

## 0. Antes da primeira vez (fora do notebook)

- Secret **`SMTP_FINANCEIRO`** no Colab (🔑 na barra lateral) com a senha de app da conta que envia,
  com acesso liberado a este notebook. É o mesmo da campanha de unificação.
- A planilha precisa estar compartilhada com a conta Google que você usa no Colab.

## 1. Instalação e conexão

Se atualizar o código com a sessão aberta, reinicie a sessão (Ambiente de execução → Reiniciar sessão).

In [ ]:
!test -d comunicacoes_financeiras && git -C comunicacoes_financeiras pull -q || git clone -q https://github.com/yagoppereira/comunicacoes_financeiras.git
!pip install -q -r comunicacoes_financeiras/requirements.txt
import sys; sys.path.insert(0, "comunicacoes_financeiras")

import pandas as pd
from comunicacoes_financeiras import fluxo, envio, titulos
from comunicacoes_financeiras.campanha import Campanha
from comunicacoes_financeiras.modelos import black_friday
from google.colab import drive, userdata

bq = fluxo.conectar_colab()
drive.mount("/content/drive")

## 2. Configuração da campanha

Mantenha o **mesmo `id_campanha`** até terminar: é ele que impede envio em dobro.

In [ ]:
PARAM = dict(id_campanha="BF-2026-11", modelo="black_friday",
             parametros={"percentual": "20", "prazo": "28/11/2026"})

PLANILHA = "https://docs.google.com/spreadsheets/d/1SJrrT__Sr0znJHiDvy9XomEJL7BNaHVEeKwFeqIjcEQ/edit"
ABA = "resumo_"

PASTA = "/content/drive/MyDrive/comunicacoes_financeiras"
PREVIA = f"{PASTA}/previa_{PARAM['id_campanha']}.csv"
registro = envio.RegistroCSV(f"{PASTA}/log.csv")

## 3. Ler a planilha

Só entram as linhas com `Considerar?` = Sim. A leitura para se houver código repetido com
saldos diferentes, linha sem código ou saldo zerado.

In [ ]:
linhas = fluxo.ler_tabela_planilha(PLANILHA, ABA, black_friday.COLUNAS_PLANILHA)
lista, dados = black_friday.selecionar(linhas)

## 4. Conferir e usar o saldo do DW — não envia nada

Compara o `Total Saldo` da planilha com os títulos vencidos em aberto no CIGAM hoje e mostra
quem mudou (pagou parte, pagou tudo, ou venceu título novo). Em seguida o saldo de cada
cliente passa a ser o do **DW**, que é o estado atual; quem zerou no DW sai do envio.
Para tirar alguém por outro motivo, ponha o código em `EXCLUIR`.

In [ ]:
conf = titulos.conferir_saldos(bq, dados, fluxo.PROJECT_ID)
conf[conf.situacao != "confere"]

In [ ]:
dados = titulos.usar_saldo_dw(dados, conf)     # o e-mail passa a usar o saldo do DW

EXCLUIR = set()          # ex.: {"004958"}
lista = [c for c in lista if c not in EXCLUIR]

## 5. Preparar — não envia nada

Busca os e-mails no DW, junta cadastros com os mesmos destinatários e salva a prévia no Drive,
com o saldo do DW de cada cliente congelado. A tabela no fim mostra quem **não** vai receber e por quê.

In [ ]:
c = Campanha.ensaio(**PARAM)
previa, resolucao = fluxo.preparar(bq, c, lista, dados=dados)
fluxo.salvar_previa(previa, PREVIA)

pd.DataFrame(resolucao.rejeitados, columns=["entrada", "motivo"])

## 6. Conferir a prévia

Um e-mail por linha, com saldo, desconto e valor à vista. `qtd_cadastros` > 1 = um e-mail
que cobre vários cadastros com os mesmos destinatários.

In [ ]:
black_friday.resumo_previa(previa, c)

In [ ]:
fluxo.previa_email(c, previa)                               # o primeiro da fila
fluxo.previa_email(c, previa[previa.qtd_cadastros > 1])    # um que junta vários cadastros
# fluxo.previa_email(c, previa, codigo="3262")             # um cliente específico

## 7. Teste — vai só para você

⚠️ Use **só uma amostra** da prévia. Confira valores, remetente e cabeçalho; abra também no celular.

In [ ]:
enviar = fluxo.criar_enviador(
    usuario="contato.financeiro@ctasmart.com.br",   # ou o seu e-mail, se a caixa for compartilhada
    senha_app=userdata.get("SMTP_FINANCEIRO"))

In [ ]:
amostra = pd.concat([previa[previa.qtd_cadastros == 1].head(2),
                     previa[previa.qtd_cadastros > 1].head(1)])

fluxo.disparar(Campanha.teste(**PARAM, para="yago.gutterres@ctasmart.com.br"),
               amostra, enviar, registro)

## 8. Produção — envia para os clientes

Só depois de conferir a prévia e o teste. Não tem desfazer. A oferta vale até o prazo da
seção 2: envie com antecedência.

In [ ]:
fluxo.disparar(Campanha.producao(**PARAM), previa, enviar, registro)

## 9. Retomar — se a sessão cair ou o limite diário segurar parte da fila

Rode as seções **1**, **2** e a célula do `enviar = ...` da seção **7**, depois estas.

In [ ]:
previa = fluxo.ler_previa(PREVIA)
fluxo.disparar(Campanha.ensaio(**PARAM), previa, None, registro)   # o que ainda falta, sem enviar

In [ ]:
fluxo.disparar(Campanha.producao(**PARAM), previa, enviar, registro)

## 10. Acompanhar

In [ ]:
log = pd.read_csv(f"{PASTA}/log.csv", dtype=str)
log = log[(log.id_campanha == PARAM["id_campanha"]) & (log.modo == "PRODUCAO")]
display(log.status.value_counts())
log[log.status == "FALHA"][["codigos", "nome", "erro"]]